# Setup: lerobot-ac-mlp 환경 설정

> **모든 노트북 실행 전 반드시 이 노트북을 먼저 1회 실행하세요.**

### 실행 순서
`setup.ipynb` → `train.ipynb` → `results.ipynb` (→ 필요시 `git.ipynb`)

### 처음 사용 시 바꿔야 할 것 (셀 0번)
| 변수 | 설명 | 예시 |
|------|------|------|
| `GIT_BRANCH` | 본인 브랜치 이름 | `"ecd-eunji"` |
| `GITHUB_TOKEN` | 본인 GitHub 토큰 | `"ghp_xxxx..."` |

### GitHub 토큰 발급 방법
1. GitHub 로그인 → 우측 상단 프로필 → **Settings**
2. 좌측 맨 아래 **Developer settings** → **Personal access tokens** → **Tokens (classic)**
3. **Generate new token (classic)** 클릭
4. Note: 아무 이름 / Expiration: 적당히 설정
5. **Scopes**: `repo` 체크 (전체)
6. **Generate token** → `ghp_`로 시작하는 토큰 복사 (이 화면 벗어나면 다시 못 봄!)
7. 아래 셀 0의 `GITHUB_TOKEN`에 붙여넣기

### 각 셀 역할
| 셀 | 역할 | 소요시간 |
|----|------|----------|
| 0 | 경로/변수 설정 | 즉시 |
| 1 | 가상환경 생성 | 최초 1분 |
| 2 | GPU 확인 | 즉시 |
| 3 | Git 클론/업데이트 | 1~2분 |
| 4 | LeRobot 설치 | 3~5분 |
| 5a~5c | 추가 패키지 설치 | 5~15분 |
| 6 | Miniconda + MesaLib | 3~5분 |
| 7 | 렌더링 테스트 | 즉시 |
| 8 | 설치 검증 | 즉시 |

---
## 0. 경로 설정

**아래 두 줄을 본인 정보로 수정하세요:**
- `GIT_BRANCH`: 본인 브랜치 (예: `"ecd-eunji"`)
- `GITHUB_TOKEN`: 위에서 발급받은 토큰

나머지는 건드리지 않아도 됩니다.

In [ ]:
# 사용자 설정
BASE_DIR = "lerobot_project"
VENV_NAME = "lerobot_env"
PYTHON_VERSION = "3.12"
GIT_REPO_URL = "https://github.com/SungwooPark5/lerobot-ac-mlp.git"
GIT_BRANCH = "ecd-eunji"
REPO_NAME = "lerobot-ac-mlp"
GITHUB_TOKEN = "***"

# 자동 설정
import os
os.environ['MPLBACKEND'] = 'Agg'
home_dir = os.path.expanduser('~')
base_path = os.path.join(home_dir, BASE_DIR)
venv_path = os.path.join(base_path, VENV_NAME)
site_packages = os.path.join(venv_path, 'lib', f'python{PYTHON_VERSION}', 'site-packages')
python_exec = os.path.join(venv_path, 'bin', 'python')
pip_exec = os.path.join(venv_path, 'bin', 'pip')
project_dir = os.path.join(base_path, REPO_NAME)
GIT_AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/SungwooPark5/lerobot-ac-mlp.git"

print("=" * 60)
print(f"Venv:    {venv_path}")
print(f"Project: {project_dir}")
print("=" * 60)

Venv:    /home1/eunji24/lerobot_project/lerobot_env
Project: /home1/eunji24/lerobot_project/lerobot-ac-mlp


## 1. 가상환경

Python 패키지를 격리해서 설치하기 위한 가상환경을 만듭니다.
- 최초 1회만 생성, 이후 실행 시 "OK"만 출력됨
- 수정할 것 없음

In [2]:
import sys
if not os.path.exists(venv_path):
    os.makedirs(base_path, exist_ok=True)
    !python3 -m venv {venv_path}
    !{pip_exec} install --upgrade pip setuptools wheel -q
    print("가상환경 생성 완료")
else:
    print(f"가상환경 OK: {venv_path}")
if site_packages not in sys.path:
    sys.path.insert(0, site_packages)

가상환경 OK: /home1/eunji24/lerobot_project/lerobot_env


## 2. GPU 확인

서버에 GPU가 잡혀 있는지 확인합니다.
- `CUDA Not Available` → 서버/큐 설정 확인 필요
- 수정할 것 없음

In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA Not Available")

ImportError: Failed to load PyTorch C extensions:
    It appears that PyTorch has loaded the `torch/_C` folder
    of the PyTorch repository rather than the C extensions which
    are expected in the `torch._C` namespace. This can occur when
    using the `install` workflow. e.g.
        $ python setup.py install && python -c "import torch"

    This error can generally be solved using the `develop` workflow
        $ python setup.py develop && python -c "import torch"  # This should succeed
    or by running Python from a different directory.

## 3. Git 클론

GitHub에서 코드를 다운로드(클론)합니다.
- 최초 실행: 저장소 전체 클론
- 재실행: 최신 코드로 업데이트 (pull)
- **이름/이메일을 본인 것으로 바꾸세요** (`git config user.name`, `user.email`)

In [32]:
if os.path.exists(project_dir):
    %cd {project_dir}
    !git remote set-url origin {GIT_AUTH_URL}
    !git fetch origin {GIT_BRANCH}
    !git checkout {GIT_BRANCH}
    !git pull origin {GIT_BRANCH}
    print("업데이트 완료")
else:
    os.makedirs(base_path, exist_ok=True)
    %cd {base_path}
    !git clone -b {GIT_BRANCH} {GIT_AUTH_URL} {REPO_NAME}
    print("클론 완료")

%cd {project_dir}
!git config user.name "eejjii-lib"
!git config user.email "lej0401@uos.ac.kr"
!echo "Branch: $(git branch --show-current)"
!git log --oneline -3

/home1/eunji24/lerobot_project/lerobot-ac-mlp
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
From https://github.com/SungwooPark5/lerobot-ac-mlp
 * branch            ecd-eunji  -> FETCH_HEAD
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Already on 'ecd-eunji'
Your branch is up to date with 'origin/ecd-eunji'.
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
From https://github.com/SungwooPark5/lerobot-ac-mlp
 * branch            ecd-eunji  -> FETCH_HEAD
Already up to date.
업데이트 완료
/home1/eunji24/lerobot_project/lerobot-ac-mlp
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home1/eunji24/miniconda3/lib/libt

---
## 4. LeRobot 설치 (editable mode)

ACT/ACM 정책이 포함된 LeRobot 프레임워크를 설치합니다.
- editable mode: 코드 수정 시 재설치 없이 바로 반영
- 최초 3~5분, 이후 "이미 설치됨" 출력

In [33]:
%%time
%cd {project_dir}
already = !{pip_exec} show lerobot 2>/dev/null | grep "^Name: lerobot"
if already:
    print("LeRobot 이미 설치됨")
    !{pip_exec} show lerobot 2>/dev/null | grep -E "^(Name|Version):"
else:
    !{pip_exec} install cmake ninja packaging 2>&1 | tail -3
    !{pip_exec} install -e . 2>&1 | tail -5
    print("LeRobot 설치 완료")

/home1/eunji24/lerobot_project/lerobot-ac-mlp
LeRobot 이미 설치됨
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Name: lerobot
Version: 0.4.3
CPU times: user 4.44 ms, sys: 17.6 ms, total: 22 ms
Wall time: 964 ms


## 5a. PyAV 백엔드

데이터셋의 비디오를 읽기 위한 디코더입니다. 수정할 것 없음.

In [34]:
import subprocess
r = subprocess.run([python_exec, "-c", "import av; print(av.__version__)"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"PyAV {r.stdout.strip()} OK")
else:
    !{pip_exec} install av 2>&1 | tail -3

PyAV 15.1.0 OK


## 5b. Mamba-SSM (ACM 정책용)

ACM(Mamba 기반 정책)에 필요한 패키지입니다.
- CUDA 컴파일 포함 → **최초 설치 5~10분** 소요
- 이후 "OK" 출력

In [35]:
import subprocess
r = subprocess.run([pip_exec, "show", "mamba-ssm"], capture_output=True, text=True)
if "Name: mamba-ssm" in r.stdout:
    print("mamba-ssm OK")
else:
    print("mamba-ssm 설치 중 (--no-build-isolation)...")
    !{pip_exec} install ninja packaging 2>&1 | tail -3
    !{pip_exec} install mamba-ssm --no-build-isolation 2>&1 | tail -15

print("lerobot 호환 의존성 복구...")
!{pip_exec} install "huggingface-hub>=0.34.2,<0.36.0" "transformers>=4.53.0,<5.0.0" 2>&1 | tail -5

for mod, name in [("mamba_ssm", "mamba-ssm"), ("huggingface_hub", "huggingface-hub"), ("transformers", "transformers")]:
    r = subprocess.run([python_exec, "-c", f"import {mod}; print({mod}.__version__)"], capture_output=True, text=True)
    print(f"  {name}: {r.stdout.strip() if r.returncode == 0 else 'FAIL'}")

mamba-ssm OK
lerobot 호환 의존성 복구...
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
  mamba-ssm: 2.3.1
  huggingface-hub: 0.35.3
  transformers: 4.57.6


## 5c. gym-aloha (Aloha 시뮬레이션 환경)

시뮬레이션 태스크(Transfer Cube 등)를 실행하기 위한 환경입니다. 수정할 것 없음.

In [36]:
import subprocess
r = subprocess.run([python_exec, '-c', 'import gym_aloha; print("OK")'], capture_output=True, text=True)
if r.returncode == 0:
    print("gym-aloha OK")
else:
    print("gym-aloha 설치 중...")
    !{pip_exec} install -e ".[aloha]" 2>&1 | tail -5

gym-aloha OK


## 5d. matplotlib (결과 시각화용)

results.ipynb에서 그래프를 그리기 위해 필요합니다. 수정할 것 없음.

In [37]:
import subprocess
r = subprocess.run([python_exec, "-c", "import matplotlib; print(matplotlib.__version__)"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"matplotlib {r.stdout.strip()} OK")
else:
    print("matplotlib 설치 중...")
    !{pip_exec} install matplotlib -q 2>&1 | tail -3
    print("matplotlib 설치 완료")

matplotlib 3.10.8 OK


---
## 6. Miniconda + MesaLib (시뮬레이션 렌더링용)

서버에는 모니터가 없어서 화면 렌더링(OpenGL)이 안 됩니다.
MesaLib를 설치해서 소프트웨어 렌더링(OSMesa)을 사용합니다.
- `CondaKeyError` 경고는 무시해도 됨
- 최종적으로 `mesalib OK` 나오면 성공

In [38]:
import subprocess, os

CONDA_DIR = os.path.expanduser('~/miniconda3')
CONDA_BIN = os.path.join(CONDA_DIR, 'bin', 'conda')
CONDA_LIB = os.path.join(CONDA_DIR, 'lib')

if not os.path.exists(CONDA_BIN):
    print("miniconda 설치 중...")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p {CONDA_DIR}
    !rm /tmp/miniconda.sh
else:
    print(f"miniconda OK: {CONDA_DIR}")

!{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>&1
!{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>&1
!{CONDA_BIN} config --remove channels defaults 2>&1 || true
!{CONDA_BIN} config --add channels conda-forge 2>&1

mesa = subprocess.run(['bash', '-c', f'ls {CONDA_LIB}/libOSMesa* 2>/dev/null'], capture_output=True, text=True)
if mesa.stdout.strip():
    print("mesalib OK")
else:
    print("mesalib 설치 중...")
    !{CONDA_BIN} install -y --override-channels -c conda-forge mesalib 2>&1 | tail -10

miniconda OK: /home1/eunji24/miniconda3
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)

CondaKeyError: 'channels': value 'defaults' not present in config

/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
mesalib OK


## 7. 렌더링 테스트

MuJoCo 시뮬레이터가 렌더링 가능한지 확인합니다.
- `egl: OK` 또는 `osmesa: OK` → 성공
- 둘 다 FAIL → 6번 셀부터 재실행

In [39]:
import subprocess, os

CONDA_DIR = os.path.expanduser('~/miniconda3')
CONDA_LIB = os.path.join(CONDA_DIR, 'lib')

lib_dirs = {CONDA_LIB}
r = subprocess.run(['bash', '-c', f'find {CONDA_DIR} -name "libOSMesa*" -o -name "libEGL*" 2>/dev/null'],
                    capture_output=True, text=True, timeout=15)
for line in r.stdout.strip().split('\n'):
    if line.strip(): lib_dirs.add(os.path.dirname(line.strip()))

ld_path = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ld_path

render_test = (
    "import os; os.environ['MUJOCO_GL']='{b}'; import mujoco; "
    "m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos=\"0 0 1\"/>"
    "<geom type=\"sphere\" size=\"0.1\"/></worldbody></mujoco>'); "
    "d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); mujoco.mj_forward(m,d); "
    "r.update_scene(d); img=r.render(); print(f'OK {{img.shape}}')"
)

for backend in ['egl', 'osmesa']:
    env = {**os.environ, 'MUJOCO_GL': backend, 'LD_LIBRARY_PATH': ld_path}
    r = subprocess.run([python_exec, '-c', render_test.format(b=backend)], capture_output=True, text=True, env=env)
    if 'OK' in r.stdout:
        print(f"{backend}: OK  <-- simulation.ipynb에서 사용")
        break
    else:
        print(f"{backend}: FAIL")

egl: FAIL
osmesa: OK  <-- simulation.ipynb에서 사용


## 8. 설치 검증

모든 필수 모듈이 import 가능한지 최종 확인합니다.
- 전부 `OK` → setup 완료, train.ipynb로 이동
- `FAIL` → 해당 패키지 설치 셀 재실행

In [40]:
import importlib, sys

src_path = os.path.join(project_dir, 'src')
if site_packages not in sys.path: sys.path.insert(0, site_packages)
if src_path not in sys.path: sys.path.insert(0, src_path)
for m in list(sys.modules.keys()):
    if m.startswith('lerobot'): del sys.modules[m]

for mod in ["lerobot", "lerobot.policies.act.configuration_act",
            "lerobot.policies.acm.configuration_acm", "lerobot.scripts.lerobot_train",
            "lerobot.scripts.lerobot_eval"]:
    try:
        importlib.import_module(mod)
        print(f"  OK: {mod}")
    except Exception as e:
        print(f"  FAIL: {mod} -> {e}")

Overwriting feature type 'VideoFrame' (VideoFrame -> VideoFrame)


  OK: lerobot
  OK: lerobot.policies.act.configuration_act
  OK: lerobot.policies.acm.configuration_acm
  OK: lerobot.scripts.lerobot_train
  OK: lerobot.scripts.lerobot_eval


---
## Setup 완료

모든 셀이 에러 없이 실행되었으면 환경 설정이 끝난 것입니다.

In [41]:
%cd {project_dir}
print("=" * 60)
print("Setup 완료!")
print("=" * 60)
!git branch --show-current
print()
print("다음 단계:")
print("  1. train.ipynb   -> ACT/ACM 학습 + 평가")
print("  2. results.ipynb -> 결과 시각화 + 다운로드")
print("  3. git.ipynb     -> Git 관리 (커밋/푸시/브랜치)")

/home1/eunji24/lerobot_project/lerobot-ac-mlp
Setup 완료!
/bin/bash: /home1/eunji24/miniconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ecd-eunji

다음 단계:
  1. train.ipynb   -> ACT/ACM 학습 + 평가
  2. results.ipynb -> 결과 시각화 + 다운로드
  3. git.ipynb     -> Git 관리 (커밋/푸시/브랜치)


/myubai/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]
